# Exercise 1: Template exercise

*Morphological Structure · version 0.1.0*

> Analyse *kitaplarımıza* as *kitap-lar-ımız-a*. Write a proposed UD feature bundle, then run the analyser of the Hands-On section and compare. Distinguish the noun's number, the possessor's person and number, and case. Explain any mismatch, and record the toolkit version and backend.

**Setup.** `pip install "turkicnlp[hfst]==0.5.0"`. Where the compiled `hfst` package has no wheel for your Python version, install `turkicnlp==0.5.0 pyhfst` instead: the first code cell then uses the pure-Python `pyhfst` reader, which has the same lookup interface. The Apertium transducers are downloaded once with `turkicnlp.download`. With `pyhfst`, readings may be listed in a different order than with `hfst`, and the pipeline may select a different reading when several have the same weight.

**Proposed analysis.** *kitap* 'book' (the stem-final *p* stays voiceless here because *-lar* begins with a consonant) + *-lar* plural (`Number=Plur`, the books are several) + *-ımız* first-person plural possessor (`Person[psor]=1`, `Number[psor]=Plur`, 'our') + *-a* dative (`Case=Dat`). The possessor's number is a separate feature from the noun's number: 'our book' would be *kitab-ımız-a* with `Number[psor]=Plur` but no `Number=Plur`.

In [ ]:
import sys
try:
    import hfst                                   # compiled HFST, as in the chapter
except ImportError:                               # fall back to the pure-Python reader
    import pyhfst
    sys.modules["hfst"] = pyhfst

import turkicnlp
from turkicnlp import Pipeline

def pipeline(lang):
    turkicnlp.download(lang, processors=["morph"])          # Apertium transducers (once)
    return Pipeline(lang, processors=["tokenize", "morph"], tokenize_backend="rule", morph_backend="apertium")

def analyser(pipe):
    """The Apertium processor of a pipeline (private API in 0.5.0)."""
    return next(p for p in pipe._processors if hasattr(p, "_analyze"))

def readings(pipe, word, plus=False):
    """All transducer readings of one word; by default without '+' continuations (copula etc.)."""
    out = [r["raw"] for r in analyser(pipe)._analyze(word)]
    return out if plus else [r for r in out if "+" not in r]

def show(pipe, sentence):
    for w in pipe(sentence).words:
        print(f"  {w.text:<18} lemma={w.lemma or '_':<10} upos={w.upos or '_':<6} feats={w.feats or '_'}")

tur = pipeline("tur")

In [ ]:
proposed = {"Case=Dat", "Number=Plur", "Number[psor]=Plur", "Person[psor]=1"}   # kitap-lar-ımız-a

print("Transducer readings:", readings(tur, "kitaplarımıza"))
for sentence in ["kitaplarımıza", "Kitaplarımıza bakmadılar."]:
    word = next(w for w in tur(sentence).words if w.text.lower().startswith("kitaplar"))
    got = set((word.feats or "").split("|")) - {""}
    print(f"\n{sentence!r}: lemma={word.lemma} upos={word.upos} feats={word.feats}")
    print("  only in proposal:", sorted(proposed - got), " only in toolkit:", sorted(got - proposed))
print("\nTurkicNLP", turkicnlp.__version__, "| backend: apertium | hfst module:", sys.modules["hfst"].__name__)

## Worked solution

Output of `experiments/ch05_exercises/run_ch05_exercises.py` on 19 October 2026 (TurkicNLP 0.5.0, HFST 3.16.0.1, Python 3.11):

```
Transducer readings: ['kitap<n><pl><px1pl><dat>']

'kitaplarımıza': lemma=kitap upos=NOUN feats=Case=Dat|Number=Plur|Number[psor]=Plur|Person[psor]=1
  only in proposal: []  only in toolkit: []

'Kitaplarımıza bakmadılar.': lemma=kitap upos=NOUN feats=Case=Dat|Number=Plur|Number[psor]=Plur|Person[psor]=1
  only in proposal: []  only in toolkit: []

TurkicNLP 0.5.0 | backend: apertium | hfst module: hfst
```

The transducer has a single reading, `kitap<n><pl><px1pl><dat>`, and the pipeline returns exactly the proposed bundle, for the word alone and inside a sentence: `Number=Plur` for the books, `Person[psor]=1` and `Number[psor]=Plur` for 'our', and `Case=Dat`. There is no mismatch to explain for this word. What the bundle does not show is the surface segmentation: neither the boundary *kitap|lar* nor the shape of *-ımız* is recorded, so the segmentation has to be stored separately (the chapter's three levels of representation).

Record with the answer: TurkicNLP 0.5.0, Apertium backend, HFST 3.16.0.1. Because the word has only one reading, reading the transducer with `pyhfst` instead gives the same result.

## Your turn — extension

1. Repeat with *kitabımıza* 'to our book' and *kitaplarına*, which is ambiguous between 'to his/her books', 'to their books' and 'to their book'. Which readings does the transducer return, and which one does the pipeline select?
2. Run the same word through the Kazakh pipeline as *кітаптарымызға*.